In [ ]:
!pip install "protobuf<=3.20.3"

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import cv2
import random
import numpy as np
import shutil
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import joblib


import tensorflow as tf
from tensorflow.keras.utils import img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array

from keras.optimizers import Adam, RMSprop
from keras.layers import Input, Concatenate, ZeroPadding2D, BatchNormalization
from keras.layers import Dense, Dropout, Activation
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from keras.layers import BatchNormalization, ZeroPadding2D, Concatenate, Input
from keras.models import Model, load_model
from keras.preprocessing import image
from keras.callbacks import ModelCheckpoint
from keras.applications.densenet import preprocess_input
import keras.backend as K
import keras

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split

# =====================================================
# 📦 IMPORT LIBRARY
# =====================================================
from cuml.svm import SVC as cuSVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("TF version:", tf.version)
print("Keras version:", tf.keras.version)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
def densenet121_model(
    img_rows,
    img_cols,
    color_type=1,
    nb_dense_block=4,
    growth_rate=32,
    nb_filter=64,
    reduction=0.5,
    dropout_rate=0.0,
    weight_decay=1e-4,
    num_classes=None,
):
    """
    DenseNet 121 Model for Keras

    Model Schema is based on
    https://github.com/flyyufelix/DenseNet-Keras

    # Returns
        A Keras model instance.
    """

    # Handle Dimension Ordering for different backends
    # global concat_axis
    img_input = Input(shape=(img_rows, img_cols, color_type), name="data")
    # concat_axis = 3

    # From architecture for ImageNet (Table 1 in the paper)
    nb_filter = 64
    nb_layers = [6, 12, 24, 16]  # For DenseNet-121

    # Initial convolution
    # x = Conv2D(nb_filter, 7, 7, subsample=(2, 2), name='conv1', bias=False)(img_input)
    x = Conv2D(
        filters=nb_filter,
        kernel_size=(7, 7),
        strides=(2, 2),
        use_bias=False,
        name="conv1",
    )(img_input)

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)

    x = MaxPooling2D((3, 3), strides=(2, 2))(x)

    # Add dense blocks
    for block_idx in range(nb_dense_block - 1):
        stage = block_idx + 2
        x, nb_filter = dense_block(
            x,
            stage,
            nb_layers[block_idx],
            nb_filter,
            growth_rate,
            dropout_rate=dropout_rate,
        )

        # Add transition_block
        x = transition_block(x, stage, nb_filter, dropout_rate=dropout_rate)
        nb_filter = int(nb_filter)

    final_stage = stage + 1
    x, nb_filter = dense_block(
        x, final_stage, nb_layers[-1], nb_filter, growth_rate, dropout_rate=dropout_rate
    )

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1 )(x)
    x = Activation("relu")(x)

    # x_fc = GlobalAveragePooling2D(name="feature_extractor_layer")(x)
    # x_fc = Dense(1000)(x_fc)
    # x_fc = Activation('softmax')(x_fc)

    # model = Model(img_input, x_fc)

    # The method below works since pre-trained weights are stored in layers but not in the model
    x_newfc = GlobalAveragePooling2D(name="feature_extractor_layer")(x)
    x_newfc = Dense(num_classes)(x_newfc)
    x_newfc = Activation("softmax")(x_newfc)

    model = Model(img_input, x_newfc)

    # ADAM OPTIMIZER
    # adm = Adam(learning_rate=1e-4)
    # model.compile(optimizer=adm, loss='categorical_crossentropy', metrics=['accuracy'])

    # RMSprop OPTIMIZER
    RMSp = RMSprop(learning_rate=1e-4)
    model.compile(optimizer=RMSp, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

In [ ]:
def conv_block(x, stage, branch, nb_filter, dropout_rate=None):
    """Apply BatchNorm, Relu, bottleneck 1x1 Conv2D, 3x3 Conv2D, and option dropout"""

    # 1x1 Convolution (Bottleneck layer)
    inter_channel = nb_filter * 4
    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = Conv2D(inter_channel, (1, 1), use_bias=False)(x)  # ✅ argumen lama diganti
    # x = Conv2D(inter_channel, 1, 1, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    # 3x3 Convolution
    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = ZeroPadding2D((1, 1))(x)
    x = Conv2D(nb_filter, (3, 3), use_bias=False)(x)  # ✅ ganti format argumen
    # x = Conv2D(nb_filter, 3, 3, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    return x


def transition_block(x, stage, nb_filter, dropout_rate=None):
    """Apply BatchNorm, 1x1 Convolution, averagePooling, optional compression, dropout"""

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = Conv2D(int(nb_filter), (1, 1), use_bias=False)(x)  # ✅ format baru
    # x = Conv2D(int(nb_filter), 1, 1, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    x = AveragePooling2D((2, 2), strides=(2, 2))(x)

    return x


def dense_block(
    x, stage, nb_layers, nb_filter, growth_rate, dropout_rate=None, grow_nb_filters=True
):
    """Build a dense_block where the output of each conv_block is fed to subsequent ones
    # Arguments
        x: input tensor
        stage: index for dense block
        nb_layers: the number of layers of conv_block to append to the model.
        nb_filter: number of filters
        growth_rate: growth rate
        grow_nb_filters: flag to decide to allow number of filters to grow
    """

    concat_feat = x

    for i in range(nb_layers):
        branch = i + 1
        x = conv_block(concat_feat, stage, branch, growth_rate, dropout_rate)
        concat_feat = Concatenate(axis=-1)([concat_feat, x])

        if grow_nb_filters:
            nb_filter += growth_rate

    return concat_feat, nb_filter

In [ ]:
data_path = (
    "/kaggle/input/datasets/dewamardana/dataset-manual-selection/dataset_centralcrop"
)

images = []
labels = []

for subfolder in os.listdir(data_path):

    subfolder_path = os.path.join(data_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for image_filename in os.listdir(subfolder_path):
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)

        labels.append(subfolder)

data = pd.DataFrame({"image": images, "label": labels})
data.head()
data.shape

In [ ]:
strat = data["label"]
train_df, dummy_df = train_test_split(
    data, train_size=0.80, shuffle=True, random_state=123, stratify=strat
)

strat = dummy_df["label"]
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True, random_state=123, stratify=strat
)

print("Training set shape:", train_df.shape)
print("Validation set shape:", valid_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
batch_size = 16
img_size = (256, 256)
channels = 3
img_shape = (img_size[0], img_size[1], channels)


tr_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

ts_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
)

train_gen = tr_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    batch_size=batch_size,
)

valid_gen = ts_gen.flow_from_dataframe(
    valid_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

test_gen = ts_gen.flow_from_dataframe(
    test_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

feat_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
)

train_gen_noaug = feat_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="sparse",
    shuffle=False,
    batch_size=batch_size,
)

In [ ]:
from keras.callbacks import ModelCheckpoint

if __name__ == "__main__":

    # Example to fine-tune on 3000 samples from Cifar10

    img_rows, img_cols = 256, 256  # Resolution of inputs
    channel = 3
    num_classes = 12
    nb_epoch = 16

    # Load Cifar10 data. Please implement your own load_data() module for your own dataset
    # X_train, Y_train, X_valid, Y_valid = load_data()

    # Load our model    print("Devices:", tf.config.list_physical_devices())

    model = densenet121_model(
        img_rows=img_rows,
        img_cols=img_cols,
        color_type=channel,
        num_classes=num_classes,
    )
    filepath = "densenet121_skenario2.keras"
    checkpoint = ModelCheckpoint(
        filepath, monitor="val_accuracy", verbose=1, save_best_only=True, mode="max"
    )
    callbacks_list = [checkpoint]
    # Start CNN

    # =====================================================
    # TRAINING CNN START
    # =====================================================
    cnn_training_start = time.time()
    history = model.fit(
        train_gen,
        epochs=nb_epoch,
        shuffle=False,
        verbose=1,
        validation_data=valid_gen,
        callbacks=callbacks_list,
    )

    # =====================================================
    # TRAINING CNN END
    # =====================================================
    cnn_training_end = time.time()

    cnn_training_time = cnn_training_end - cnn_training_start

    print(f"\nCNN Training Time: {cnn_training_time:.2f} seconds")

    # =====================================================
    # SAVE CNN MODEL
    # =====================================================
    cnn_model_path = "densenet121_skenario2.keras"

    model.save(cnn_model_path)

In [ ]:
# =============================
# 1. Evaluasi Train & Validation (dari history)
# =============================
train_acc = history.history["accuracy"][-1]
train_loss = history.history["loss"][-1]
val_acc = history.history["val_accuracy"][-1]
val_loss = history.history["val_loss"][-1]

print("=== TRAINING METRICS ===")
print("Training Accuracy :", train_acc)
print("Training Loss     :", train_loss)
print("\n=== VALIDATION METRICS ===")
print("Validation Accuracy :", val_acc)
print("Validation Loss     :", val_loss)

In [ ]:
# =============================
# 2. Plot Training vs Validation Accuracy & Loss
# =============================
import matplotlib.pyplot as plt

epochs = range(len(history.history["accuracy"]))

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Accuracy")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["loss"], label="Training Loss")
plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Loss")
plt.show()

In [ ]:
from tensorflow.keras.utils import to_categorical

# =============================
# 3. Prediksi Validation Set
# =============================
ypred = model.predict(valid_gen, verbose=1)

# ground truth
ytrue = valid_gen.classes
ytrue_cat = to_categorical(ytrue, num_classes=num_classes)

In [ ]:
# =============================
# 4. Evaluasi Test Dataset
# =============================
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print("\n=== TEST METRICS ===")
print("Test Accuracy :", test_acc)
print("Test Loss     :", test_loss)

In [ ]:
# =============================
# 5. Akurasi Manual
# =============================
predicted_labels = np.argmax(ypred, axis=1)
accurate = np.sum(predicted_labels == ytrue)
total = len(ytrue)

print("\n=== MANUAL ACCURACY CHECK ===")
print("Total Data     :", total)
print("Correct Predict:", accurate)
print("Wrong Predict  :", total - accurate)
print("Accuracy (%)   :", accurate / total * 100)

In [ ]:
# =============================
# 6. Log Loss
# =============================
from sklearn.metrics import log_loss

val_logloss = log_loss(ytrue_cat, ypred)
print("\nValidation Log Loss:", val_logloss)

In [ ]:
# =============================
# 7. Confusion Matrix & Classification Report
# =============================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(ytrue, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("\n=== Classification Report ===")
print(
    classification_report(
        ytrue, predicted_labels, target_names=list(train_gen.class_indices.keys())
    )
)

In [ ]:
# model = load_model("/kaggle/input/desenet-121-mushroom-12-class/keras/default/1/bestmodel.keras")
model.summary(expand_nested=True, line_length=200)

In [ ]:
from sklearn.preprocessing import StandardScaler
import gc

# =====================================================
# ⚙️ KONFIGURASI DASAR
# =====================================================
img_size = (256, 256)
channels = 3

# Ambil output dari layer global average pooling terakhir
feature_model = Model(
    inputs=model.input, outputs=model.get_layer("feature_extractor_layer").output
)


# =====================================================
# 🔍 EKSTRAKSI FITUR
# =====================================================
def extract_features(generator, feature_extractor):
    features = feature_extractor.predict(generator, verbose=1)
    labels = np.array(generator.classes)
    return features, labels


# Ekstraksi fitur
print("Ekstraksi fitur training...")
train_features, train_labels = extract_features(train_gen_noaug, feature_model)

print("Ekstraksi fitur testing...")
test_features, test_labels = extract_features(test_gen, feature_model)

# # =====================================================
# # 🔹 Encode label ke integer
# # =====================================================
train_labels = train_gen_noaug.classes
test_labels = test_gen.classes


# =====================================================
# ⚖️ NORMALISASI FITUR
# =====================================================
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# =====================================================
# 🔹 HAPUS OBJEK TIDAK DIPAKAI UNTUK HEMAT MEMORI
# =====================================================
gc.collect()

In [ ]:
from sklearn.svm import SVC  # CPU

# Jika Linear
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============================================
# 1. Inisialisasi model SVM
# ============================================
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)

# svm_model = OneVsRestClassifier(
#     LinearSVC(
#         C=1.0,
#         max_iter=10000
#     )
# )


# ============================================
# 2. Training
# ============================================
print("Training SVM sedang berjalan...\n")

svm_training_start = time.time()
svm_model.fit(train_features, train_labels)
svm_training_end = time.time()

svm_training_time = svm_training_end - svm_training_start

print(f"\nSVM Training Time: " f"{svm_training_time:.2f} seconds")

print("Training selesai!\n")

svm_model_path = "svm_model_skenario2.pkl"

joblib.dump(svm_model, svm_model_path)

In [ ]:
# =====================================================
# CLASSIFICATION START
# =====================================================
classification_start = time.time()

# =====================================================
# FEATURE EXTRACTION TEST
# =====================================================
test_features, test_labels = extract_features(test_gen, feature_model)

test_features = scaler.transform(test_features)

# =====================================================
# SVM PREDICTION
# =====================================================
svm_predictions = svm_model.predict(test_features)

# =====================================================
# CLASSIFICATION END
# =====================================================
classification_end = time.time()

total_classification_time = classification_end - classification_start

print(f"\nClassification Time: " f"{total_classification_time:.2f} seconds")

# =====================================================
# CLASSIFICATION TIME PER IMAGE
# =====================================================
total_test_images = len(test_labels)

classification_time_per_image = total_classification_time / total_test_images

print(f"Classification Time per Image: " f"{classification_time_per_image:.4f} seconds")

In [ ]:
# =====================================================
# ACCURACY
# =====================================================
accuracy = accuracy_score(test_labels, svm_predictions)

# =====================================================
# PRECISION
# =====================================================
precision = precision_score(test_labels, svm_predictions, average="weighted")

# =====================================================
# RECALL
# =====================================================
recall = recall_score(test_labels, svm_predictions, average="weighted")

# =====================================================
# F1 SCORE
# =====================================================
f1 = f1_score(test_labels, svm_predictions, average="weighted")

In [ ]:
# =====================================================
# TOTAL TRAINING TIME
# =====================================================
total_training_time = cnn_training_time + svm_training_time

# =====================================================
# CNN MODEL SIZE
# =====================================================
cnn_model_size = os.path.getsize(cnn_model_path) / (1024 * 1024)

# =====================================================
# SVM MODEL SIZE
# =====================================================
svm_model_size = os.path.getsize(svm_model_path) / (1024 * 1024)

# =====================================================
# TOTAL MODEL SIZE
# =====================================================
total_model_size = cnn_model_size + svm_model_size

In [ ]:
# =====================================================
# FINAL RESULT
# =====================================================
print("\n===================================")
print("FINAL RESULT - SKEMA 2")
print("===================================")

print(f"Accuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1-Score               : {f1:.4f}")

print(f"\nCNN Training Time (s) : {cnn_training_time:.2f}")
print(f"SVM Training Time (s) : {svm_training_time:.2f}")

print(f"Total Training Time(s): {total_training_time:.2f}")

print(f"\nClassification Time(s): " f"{total_classification_time:.2f}")

print(f"\nCNN Model Size (MB)   : {cnn_model_size:.2f}")
print(f"SVM Model Size (MB)   : {svm_model_size:.2f}")

print(f"Total Model Size(MB)  : {total_model_size:.2f}")

In [ ]:
# =====================================================
# CONFUSION MATRIX
# =====================================================
cm = confusion_matrix(test_labels, svm_predictions)

plt.figure(figsize=(10, 8))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# ============================================
# Prediksi pada data test
# ============================================
svm_predictions = svm_model.predict(test_features)
# svm_probabilities = svm_model.predict_proba(test_features)[:, 11]

# ============================================
# Evaluasi
# ============================================
acc = accuracy_score(test_labels, svm_predictions)
print("SVM Model Accuracy: {:.2f}%".format(acc * 100))

print("\n📌 Classification Report:")
print(classification_report(test_labels, svm_predictions))

print("\n📌 Confusion Matrix:")
print(confusion_matrix(test_labels, svm_predictions))

# =====================================================
# CLASSIFICATION REPORT
# =====================================================
print("\n=== Classification Report ===")

print(
    classification_report(
        test_labels, svm_predictions, target_names=list(test_gen.class_indices.keys())
    )
)

In [ ]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.svm import SVC
# import numpy as np
# import gc

# from sklearn.svm import LinearSVC
# from sklearn.multiclass import OneVsRestClassifier

# # =====================================================
# # ⚙️ PERSIAPAN K-FOLD
# # =====================================================
# X = data["image"].values   # path gambar
# y = data["label"].values   # label

# k = 5
# skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=123)

# all_scores = []
# fold_no = 1

# # =====================================================
# # 🔄 LOOP K-FOLD
# # =====================================================
# for train_index, test_index in skf.split(X, y):

#     print(f"\n==============================")
#     print(f"        FOLD {fold_no}")
#     print(f"==============================")

#     # Buat dataframe train/test per fold
#     train_df = data.iloc[train_index].reset_index(drop=True)
#     test_df  = data.iloc[test_index].reset_index(drop=True)

#     # -------------------------------------------------
#     # 1. Buat generator untuk fold ini
#     # -------------------------------------------------
#     train_gen_fold = feat_gen.flow_from_dataframe(
#         train_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     test_gen_fold = ts_gen.flow_from_dataframe(
#         test_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     # -------------------------------------------------
#     # 2. Ekstraksi fitur CNN untuk fold ini
#     # -------------------------------------------------
#     print("Ekstraksi fitur training...")
#     train_features, train_labels = extract_features(train_gen_fold, feature_model)

#     print("Ekstraksi fitur test...")
#     test_features, test_labels = extract_features(test_gen_fold, feature_model)

#     # -------------------------------------------------
#     # 3. Normalisasi
#     # -------------------------------------------------
#     scaler = StandardScaler()
#     train_features = scaler.fit_transform(train_features)
#     test_features  = scaler.transform(test_features)

#     # -------------------------------------------------
#     # 4. Train SVM
#     # -------------------------------------------------
#     # svm = SVC(kernel="rbf", C=1.0, gamma="scale")
#     # svm.fit(train_features, train_labels)

#     svm = OneVsRestClassifier(
#         LinearSVC(
#             C=1.0,
#             max_iter=10000
#         )
#     )
#     svm.fit(train_features, train_labels)

#     # -------------------------------------------------
#     # 5. Evaluasi
#     # -------------------------------------------------
#     preds = svm.predict(test_features)

#     acc = accuracy_score(test_labels, preds)
#     all_scores.append(acc)

#     print(f"Accuracy Fold {fold_no}: {acc*100:.2f}%\n")
#     print(classification_report(test_labels, preds))
#     print(confusion_matrix(test_labels, preds))

#     fold_no += 1
#     gc.collect()


# # =====================================================
# # 📊 HASIL FINAL
# # =====================================================
# print("\n================================")
# print("        FINAL K-FOLD RESULT")
# print("================================")
# for i, score in enumerate(all_scores, start=1):
#     print(f"Fold {i}: {score*100:.2f}%")

# print("\nAverage Accuracy:", np.mean(all_scores)*100, "%")